# 8-bob. Zararli dasturlarni aniqlash II: dinamik tahlil va chuqur o'qitish
## Amaliy misol: fayllarni tasvirga aylantirish va oilalarga ajratish

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/08-bob/8-3_fayllarni-tasvirga-aylantirish-va-oilalarga.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 8-bob.*

Quyidagi dasturda 8 ta zararli dastur oilasiga mansub sun'iy fayllar yaratiladi. Har bir oila to'rt xil bo'lim turining o'ziga xos tartibi va ulushlaridan tashkil topgan: nollar bilan to'ldirilgan bo'lim, matnli satrlar, kod va shifrlangan qism. Har bir nusxada bo'limlar hajmi tasodifiy o'zgaradi, baytlarning bir qismi tasodifan almashtiriladi va fayl boshiga turli uzunlikdagi qo'shimcha kod qo'shiladi, natijada barcha bo'limlar siljiydi. Fayllar 64 bayt kenglikdagi qatorlarga bo'linib, 32×32 o'lchamli kulrang tasvirga aylantiriladi. So'ngra ikki usul taqqoslanadi: piksellar bo'yicha eng yaqin qo'shnilar va kichik konvolyutsion neyron tarmoq. Ular oddiy sinov to'plamida va kuchliroq o'zgartirilgan "yangi variantlar"da sinaladi. Oxirida o'qitish to'plamiga o'zgartirilgan nusxalar qo'shilib, tajriba takrorlanadi. Dastur PyTorch kutubxonasini talab qiladi, u Google Colab muhitida oldindan o'rnatilgan. Dastur amaliy mashg'ulotning A8 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
import torch
from torch import nn
from PIL import Image
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
OILALAR = 8                                     # 8 ta zararli dastur oilasi (sun'iy)

# Fayl bo'limlarining to'rt turi va ulardagi baytlar taqsimoti
kod_p = np.full(256, 0.2); kod_p[[0x00, 0x8B, 0x89, 0xE8, 0xFF, 0x48, 0x83, 0x0F]] = 40; kod_p /= kod_p.sum()
def bolim(tur, n):
    if tur == "nol":         return np.where(rng.random(n) < 0.97, 0, rng.integers(0, 256, n))
    if tur == "matn":        return rng.integers(32, 127, n)                   # o'qiladigan belgilar
    if tur == "kod":         return rng.choice(256, n, p=kod_p)                # tez-tez uchraydigan buyruqlar
    if tur == "shifrlangan": return rng.integers(0, 256, n)                    # paketlangan qism

# Har bir oila bo'limlarning o'ziga xos tartibi va ulushlariga ega
tuzilish = [[(rng.choice(["nol", "matn", "kod", "shifrlangan"]), rng.uniform(0.05, 0.4))
             for _ in range(rng.integers(4, 7))] for _ in range(OILALAR)]

def fayl(oila, ozgarish=0.05, siljish=0.15):
    """Bitta zararli fayl: oila tuzilishi + har bir nusxaga xos tasodifiy o'zgarishlar."""
    uzunlik = rng.integers(8_000, 16_000)
    jami = sum(u for _, u in tuzilish[oila])
    bosh = [bolim("kod", int(uzunlik * rng.uniform(0, siljish)))]   # boshiga qo'shilgan kod: bo'limlar siljiydi
    b = np.concatenate(bosh + [bolim(tur, int(uzunlik * u / jami * rng.uniform(0.6, 1.4)))
                               for tur, u in tuzilish[oila]]).astype(np.uint8)
    maska = rng.random(len(b)) < ozgarish                      # polimorf o'zgarishlar
    b[maska] = rng.integers(0, 256, maska.sum())
    return b

def tasvir(b, en=64, olcham=32):
    """Baytlarni kulrang tasvirga aylantirish: har bir bayt - bitta piksel (0 qora, 255 oq)."""
    b = b[: len(b) // en * en].reshape(-1, en)
    return np.asarray(Image.fromarray(b).resize((olcham, olcham), Image.BILINEAR), dtype=np.float32) / 255

def toplam(n, ozgarish=0.05, siljish=0.15):
    X = np.stack([tasvir(fayl(o, ozgarish, siljish)) for o in range(OILALAR) for _ in range(n)])
    y = np.repeat(np.arange(OILALAR), n)
    return X, y

X_o, y_o = toplam(200)
X_s, y_s = toplam(60)
X_yangi, y_yangi = toplam(60, ozgarish=0.15, siljish=0.4)      # kuchliroq o'zgartirilgan yangi variantlar

def cnn_oqitish(X, y):
    """Kichik konvolyutsion neyron tarmoqni o'qitish."""
    torch.manual_seed(0)
    cnn = nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 32x32 -> 16x16
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 16x16 -> 8x8
        nn.Flatten(), nn.Linear(16 * 8 * 8, 32), nn.ReLU(), nn.Linear(32, OILALAR))
    opt = torch.optim.Adam(cnn.parameters(), lr=0.003)
    Xt, yt = torch.tensor(X).unsqueeze(1), torch.tensor(y)
    for davr in range(20):
        for i in torch.randperm(len(Xt)).split(32):
            opt.zero_grad()
            nn.functional.cross_entropy(cnn(Xt[i]), yt[i]).backward()
            opt.step()
    return lambda X: cnn(torch.tensor(X).unsqueeze(1)).argmax(1).numpy()

def knn_oqitish(X, y):
    """Taqqoslash uchun: piksellar bo'yicha eng yaqin qo'shnilar usuli."""
    knn = KNeighborsClassifier(3).fit(X.reshape(len(X), -1), y)
    return lambda X: knn.predict(X.reshape(len(X), -1))

# Boyitilgan o'qitish to'plami: kuchli o'zgartirilgan nusxalar ham qo'shiladi
X_q, y_q = toplam(200, ozgarish=0.15, siljish=0.4)
X_b, y_b = np.concatenate([X_o, X_q]), np.concatenate([y_o, y_q])

print(f"{'Model':<42}{'sinov':>8}{'yangi variantlar':>18}")
for nom, X, y in [("oddiy to'plam", X_o, y_o), ("boyitilgan to'plam", X_b, y_b)]:
    for usul, oqit in [("Eng yaqin qo'shnilar", knn_oqitish), ("Konvolyutsion tarmoq", cnn_oqitish)]:
        model = oqit(X, y)
        print(f"{usul + ', ' + nom:<42}{(model(X_s) == y_s).mean():>8.1%}"
              f"{(model(X_yangi) == y_yangi).mean():>18.1%}")

# Har bir oiladan bitta namunaning tasvirini saqlaymiz (darslikdagi rasm uchun)
Image.fromarray(np.hstack([(X_s[o * 60] * 255).astype(np.uint8) for o in range(5)])) \
     .resize((5 * 128, 128), Image.NEAREST).save("oilalar.png")

Model                                        sinov  yangi variantlar


Eng yaqin qo'shnilar, oddiy to'plam         100.0%             85.4%


Konvolyutsion tarmoq, oddiy to'plam         100.0%             71.7%
Eng yaqin qo'shnilar, boyitilgan to'plam    100.0%            100.0%


Konvolyutsion tarmoq, boyitilgan to'plam    100.0%            100.0%


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Model                                        sinov  yangi variantlar
Eng yaqin qo'shnilar, oddiy to'plam         100.0%             85.4%
Konvolyutsion tarmoq, oddiy to'plam         100.0%             71.7%
Eng yaqin qo'shnilar, boyitilgan to'plam    100.0%            100.0%
Konvolyutsion tarmoq, boyitilgan to'plam    100.0%            100.0%
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 8-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.